<a href="https://colab.research.google.com/github/pinkprincess536/eeg/blob/main/preprocess.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<a href="https://colab.research.google.com/github/pinkprincess536/eeg/blob/main/preprocess.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
!pip install mne numpy scikit-learn

In [2]:
import mne
import numpy as np
import os, re, subprocess, time
from concurrent.futures import ThreadPoolExecutor, as_completed

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!ls /content/drive/MyDrive/EEG_PROJECT/processed/*.npy

## Cell 1: Config

All tunable parameters. Seed=42 for reproducible split. Subset to handle RAM.

In [ ]:
BASE_DIR = "/content/drive/MyDrive/EEG_PROJECT"
os.makedirs(BASE_DIR, exist_ok=True)

# All 24 patients, serial order, shuffled with seed=42
# All 24 patients | change to range(1,12) for quick runs
PATIENT_RANGE = range(1, 25)
ALL_PATIENTS = [f"chb{i:02d}" for i in PATIENT_RANGE]
np.random.seed(42)
shuffled = ALL_PATIENTS.copy()
np.random.shuffle(shuffled)
TRAIN_PATIENTS = sorted(shuffled[:9])
TEST_PATIENTS  = sorted(shuffled[9:])

print(f"All {len(ALL_PATIENTS)} patients: {ALL_PATIENTS}")
print(f"Train ({len(TRAIN_PATIENTS)}): {TRAIN_PATIENTS}")
print(f"Test  ({len(TEST_PATIENTS)}): {TEST_PATIENTS}")

# Pipeline params
WINDOW_SIZE_SEC = 7.0
OVERLAP         = 0.3
LOWCUT          = 0.5
HIGHCUT         = 40.0
NOTCH_FREQ      = 60.0

# Balancing
NON_SEIZURE_SAMPLES  = 500

# Test set: larger sample for realistic unbalanced evaluation
NON_SEIZURE_SAMPLES_TEST = None  # None = all non-seizure windows, no cap on scaling
RANDOM_SEED          = 42

# Subset mode: process fewer recordings for testing (set to None for full run)
MAX_RECORDINGS_PER_PATIENT = None  # None = all | 3 = first 3 recordings per patient

print(f"Window: {WINDOW_SIZE_SEC}s | Overlap: {OVERLAP} | Filter: {LOWCUT}-{HIGHCUT} Hz")
print(f"Non-seizure samples: {NON_SEIZURE_SAMPLES} | Max rec/patient: {MAX_RECORDINGS_PER_PATIENT or 'ALL'}")

## Cell 2: Download Data

6 parallel downloads. Skips files already on Drive.

In [ ]:
# Cell 2 (S3): Bulk download from PhysioNet AWS mirror
# 20 parallel streams, auto-splits large files into 16MB chunks
# Skips files already on Drive -- safe to re-run after timeout
!pip install -q awscli

# Tune S3 parallelism
!mkdir -p ~/.aws
with open(os.path.expanduser("~/.aws/config"), "w") as f:
    f.write("""[default]
s3 =
    max_concurrent_requests = 20
    multipart_threshold = 64MB
    multipart_chunksize = 16MB
""")

start_time = time.time()

for i in PATIENT_RANGE:
    pid = f"chb{i:02d}"
    print(f"\n=== Syncing {pid} ===")
    %time !aws s3 sync --no-sign-request s3://physionet-open/chbmit/1.0.0/{pid}/ {BASE_DIR}/{pid}/

    # Summary file lives at dataset root on S3
    summary_dst = f"{BASE_DIR}/{pid}-summary.txt"
    if not os.path.exists(summary_dst):
        !aws s3 cp --no-sign-request s3://physionet-open/chbmit/1.0.0/{pid}/{pid}-summary.txt {summary_dst} 2>/dev/null || true

elapsed = time.time() - start_time
print(f"\n===== S3 SYNC DONE in {elapsed/60:.0f} min =====")


In [ ]:
def parse_seizure_summary(summary_path):
    with open(summary_path, "r") as f:
        content = f.read()
    sections = re.split(r"File Name:\s*", content)[1:]
    sm = {}
    for s in sections:
        m = re.match(r"(\S+)", s)
        if not m: continue
        fn = m.group(1)
        st = [int(x) for x in re.findall(r"Seizure\s+Start\s+Time:\s*(\d+)\s*seconds", s)]
        en = [int(x) for x in re.findall(r"Seizure\s+End\s+Time:\s*(\d+)\s*seconds", s)]
        sm[fn] = list(zip(st, en))
    return sm

SEIZURE_MAP = {}
for p in ALL_PATIENTS:
    sp = os.path.join(BASE_DIR, f"{p}-summary.txt")
    if os.path.exists(sp):
        SEIZURE_MAP[p] = parse_seizure_summary(sp)
        n_seiz = sum(1 for v in SEIZURE_MAP[p].values() if v)
        print(f"{p}: {len(SEIZURE_MAP[p])} recordings, {n_seiz} with seizures")
    else:
        SEIZURE_MAP[p] = {}

## Cell 4: bandpass_filter()

Filter full recording ONCE before windowing. No per-window filtering.

In [ ]:
def bandpass_filter(raw, lowcut=0.5, highcut=40.0, notch_freq=60.0):
    rf = raw.copy()
    rf.filter(lowcut, highcut, fir_design="firwin", verbose=False)
    rf.notch_filter(freqs=notch_freq, verbose=False)
    return rf.get_data()

## Cell 5: create_windows()

30% overlap -> stride=1254 samples (7s @ 256Hz).

In [ ]:
def create_windows(signal, sfreq, window_size_sec=7.0, overlap=0.3):
    ws = int(window_size_sec * sfreq)
    st = int(ws * (1 - overlap))
    total = signal.shape[1]
    wins = []
    for s in range(0, total - ws, st):
        wins.append(signal[:, s:s+ws])
    return np.array(wins)

## Cell 6: create_labels()

Soft overlap: any part of window touches seizure -> label=1.

In [ ]:
def create_labels(windows, sfreq, seizure_intervals, overlap=0.3):
    n = windows.shape[0]
    ws = windows.shape[2]
    stride = int(ws * (1 - overlap))
    labs = np.zeros(n, dtype=int)
    if not seizure_intervals: return labs
    for i in range(n):
        a = (i*stride)/sfreq
        b = (i*stride+ws)/sfreq
        for st, en in seizure_intervals:
            if b >= st and a <= en:
                labs[i] = 1; break
    return labs

## Cell 7: process_recording()

Standardizes to 23 EEG channels. Crash-free: skips bad files.

In [ ]:
# Standard bipolar EEG channels (22 unique, EKG/EMG refs skipped)
EEG_CHANNELS = [
    "FP1-F7", "F7-T7", "T7-P7", "P7-O1",
    "FP1-F3", "F3-C3", "C3-P3", "P3-O1",
    "FP2-F4", "F4-C4", "C4-P4", "P4-O2",
    "FP2-F8", "F8-T8", "T8-P8", "P8-O2",
    "FZ-CZ", "CZ-PZ",
    "P7-T7", "T7-FT9", "FT9-FT10", "FT10-T8"
]

def process_recording(edf_path, seizure_intervals,
                      window_size_sec=7.0, overlap=0.3,
                      lowcut=0.5, highcut=40.0, notch_freq=60.0):
    try:
        raw = mne.io.read_raw_edf(edf_path, preload=True, verbose=False)
    except Exception as e:
        print(f"    SKIP {os.path.basename(edf_path)}: {e}")
        return None, None, None

    # Standardize to 23 channels (pick channels that exist in this recording)
    available = raw.ch_names
    channels_to_use = [ch for ch in EEG_CHANNELS if ch in available]
    if len(channels_to_use) < 18:
        print(f"    SKIP {os.path.basename(edf_path)}: only {len(channels_to_use)} matching channels")
        return None, None, None

    raw.pick_channels(channels_to_use)
    n_chan = len(channels_to_use)
    channel_names = channels_to_use.copy()

    signal = bandpass_filter(raw, lowcut, highcut, notch_freq)
    sfreq = raw.info["sfreq"]
    windows = create_windows(signal, sfreq, window_size_sec, overlap)

    # Pad to 23 channels if needed (zero-fill missing channels)
    if n_chan < 23:
        pad = np.zeros((23 - n_chan, windows.shape[1], windows.shape[2]), dtype=windows.dtype)
        windows = np.concatenate([windows, pad], axis=0)
        # Mark padded channels as "---MISSING---"
        channel_names += ["---MISSING---"] * (23 - n_chan)
    elif n_chan > 23:
        windows = windows[:23]
        channel_names = channel_names[:23]


    # Skip recordings too short for even one window
    if windows.ndim != 3 or windows.shape[0] == 0:
        print(f"    SKIP {os.path.basename(edf_path)}: too short")
        return None, None, None

    labels = create_labels(windows, sfreq, seizure_intervals, overlap)
    return windows, labels, channel_names

## Cell 8: preprocess_streaming() -- RAM-Safe

Processes recordings one at a time. Reservoir-samples non-seizure windows.
Never holds more than 500 + seizure_count windows in RAM.

In [ ]:
def preprocess_streaming(patient_ids, seizure_map, base_dir,
                         n_non_seizure=500, seed=42,
                         max_rec=None):
    np.random.seed(seed)

    all_seizure_wins = []
    all_seizure_labs = []

    # Reservoir for non-seizure (pre-allocated, never grows)
    non_wins = []  # list (unlimited) or pre-alloc array (limited)
    non_labs = None  # set after all non-seizure windows collected
    non_count = 0
    n_chan_expected = None

    total_recordings = 0
    skipped = 0

    for pid in patient_ids:
        pdir = os.path.join(base_dir, pid)
        if not os.path.isdir(pdir):
            continue

        edf_files = sorted([f for f in os.listdir(pdir) if f.endswith(".edf")])
        if max_rec: edf_files = edf_files[:max_rec]

        for edf in edf_files:
            intervals = seizure_map[pid].get(edf, [])
            path = os.path.join(pdir, edf)

            result = process_recording(path, intervals,
                                       window_size_sec=WINDOW_SIZE_SEC,
                                       overlap=OVERLAP)
            if result[0] is None:
                skipped += 1
                continue

            windows, labels, ch_names = result
            total_recordings += 1

            # Initialize non-seizure storage on first window
            if isinstance(non_wins, list) and len(non_wins) == 0:
                if n_non_seizure is not None:
                    n_chan = windows.shape[1]
                    non_wins = np.zeros((n_non_seizure, n_chan, windows.shape[2]), dtype=np.float32)

            # Split seizure vs non-seizure
            sez_mask = labels == 1
            non_mask = labels == 0

            # Keep ALL seizure windows
            if sez_mask.any():
                all_seizure_wins.append(windows[sez_mask])
                all_seizure_labs.append(labels[sez_mask])

            # Collect non-seizure windows
            non_idx = np.where(non_mask)[0]
            for idx in non_idx:
                non_count += 1
                if isinstance(non_wins, list):
                    non_wins.append(windows[idx])
                elif non_count <= n_non_seizure:
                    non_wins[non_count - 1] = windows[idx]
                else:
                    j = np.random.randint(0, non_count)
                    if j < n_non_seizure:
                        non_wins[j] = windows[idx]

            n_s = int(sez_mask.sum())
            n_n = int(non_mask.sum())
            print(f"  {pid}/{edf} -> {windows.shape[0]} windows ({n_s} sez, {n_n} non) | "
                  f"total sez: {sum(w.shape[0] for w in all_seizure_wins)} | "
                  f"non-seizure tracked: {non_count}")

    # Convert list to array if in unlimited mode
    if isinstance(non_wins, list):
        non_wins = np.array(non_wins) if non_wins else np.zeros((0, 1, 1792), dtype=np.float32)
        non_labs = np.zeros(len(non_wins), dtype=int)

    # Combine
    if all_seizure_wins:
        sez_win = np.concatenate(all_seizure_wins, axis=0)
        sez_lab = np.concatenate(all_seizure_labs, axis=0)
    else:
        sez_win = np.zeros((0, non_wins.shape[1], non_wins.shape[2]), dtype=np.float32)
        sez_lab = np.zeros(0, dtype=int)

    X = np.concatenate([sez_win, non_wins], axis=0)
    y = np.concatenate([sez_lab, non_labs], axis=0)
    idx = np.random.permutation(len(X))
    X, y = X[idx], y[idx]

    print(f"
Done: {total_recordings} recordings, {skipped} skipped")
    print(f"  Seizure windows: {len(sez_win)}")
    print(f"  Non-seizure windows: {len(non_wins)} (from {non_count} total)")
    print(f"  Final shape: X={X.shape}, y={y.shape}")
    return X.astype(np.float32), y, EEG_CHANNELS

## Cell 9: Preprocess Training Set

Streaming processing. 9 patients. Output stays in RAM (balanced to ~500+seizure).

In [ ]:
print("
===== PREPROCESSING TRAIN SET =====")
print(f"Patients: {TRAIN_PATIENTS}
")

X_train, y_train, ch_names = preprocess_streaming(
    TRAIN_PATIENTS, SEIZURE_MAP, BASE_DIR,
    n_non_seizure=NON_SEIZURE_SAMPLES, seed=RANDOM_SEED,
    max_rec=MAX_RECORDINGS_PER_PATIENT)

print(f"
Train ready: {X_train.shape}, {y_train.shape}")
print(f"Channels: {ch_names}")
print(f"Seizure: {np.sum(y_train)}, Non-seizure: {len(y_train)-np.sum(y_train)}")

## Cell 10: Preprocess Test Set

Streaming processing. 2 held-out patients.

In [ ]:
print("
===== PREPROCESSING TEST SET =====")
print(f"Patients: {TEST_PATIENTS}
")

X_test, y_test, _ = preprocess_streaming(
    TEST_PATIENTS, SEIZURE_MAP, BASE_DIR,
    n_non_seizure=NON_SEIZURE_SAMPLES_TEST, seed=RANDOM_SEED,
    max_rec=MAX_RECORDINGS_PER_PATIENT)

print(f"
Test ready: {X_test.shape}, {y_test.shape}")
print(f"Seizure: {np.sum(y_test)}, Non-seizure: {len(y_test)-np.sum(y_test)}")

## Cell 11: Normalize Per-Channel

Each of the 23 EEG channels normalized independently.
Stats computed from TRAIN data only, applied to both train and test.

In [ ]:
# Compute per-channel mean/std from TRAIN only
train_mean = X_train.mean(axis=(0, 2), keepdims=True)
train_std  = X_train.std( axis=(0, 2), keepdims=True)

X_train = (X_train - train_mean) / (train_std + 1e-8)
X_test  = (X_test  - train_mean) / (train_std + 1e-8)

print(f"Normalized per-channel.")
print(f"  Train shape: {X_train.shape}  (samples, channels, time)")
print(f"  Test  shape: {X_test.shape}")
print(f"  Channel means after norm: {X_train.mean(axis=(0,2))[:5]}...")

OUTPUT_DIR = os.path.join(BASE_DIR, "processed")
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Save .npy files
np.save(os.path.join(OUTPUT_DIR, "X_train.npy"), X_train)
np.save(os.path.join(OUTPUT_DIR, "y_train.npy"), y_train)
np.save(os.path.join(OUTPUT_DIR, "X_test.npy"),  X_test)
np.save(os.path.join(OUTPUT_DIR, "y_test.npy"),  y_test)

# Save normalization stats (critical for inference on new EEG)
np.save(os.path.join(OUTPUT_DIR, "train_mean.npy"), train_mean)
np.save(os.path.join(OUTPUT_DIR, "train_std.npy"),  train_std)
np.save(os.path.join(OUTPUT_DIR, "channel_names.npy"), np.array(ch_names))

# Save metadata
with open(os.path.join(OUTPUT_DIR, "info.txt"), "w") as f:
    f.write(f"Train patients ({len(TRAIN_PATIENTS)}): {TRAIN_PATIENTS}\n")
    f.write(f"Test patients  ({len(TEST_PATIENTS)}): {TEST_PATIENTS}\n")
    f.write(f"Window: {WINDOW_SIZE_SEC}s, Overlap: {OVERLAP}\n")
    f.write(f"Non-seizure samples: {NON_SEIZURE_SAMPLES}\n")
    f.write(f"Random seed: {RANDOM_SEED}\n")
    f.write(f"Max recordings per patient: {MAX_RECORDINGS_PER_PATIENT or 'ALL'}\n")
    f.write(f"Train shape: {X_train.shape}\n")
    f.write(f"Test shape:  {X_test.shape}\n")
    f.write(f"Normalization: per-channel z-score using train_mean.npy / train_std.npy\n")
    f.write(f"Channels ({len(ch_names)}): {ch_names}\n")

# Print file sizes
for fn in sorted(os.listdir(OUTPUT_DIR)):
    sz = os.path.getsize(os.path.join(OUTPUT_DIR, fn))
    print(f"  {fn}: {sz/1e6:.1f} MB")

total = sum(os.path.getsize(os.path.join(OUTPUT_DIR, f))
            for f in os.listdir(OUTPUT_DIR))
print(f"\nTotal: {total/1e6:.1f} MB saved to:")
print(f"  {OUTPUT_DIR}/")
print(f"\nNEXT: Download from Drive -> Upload to Kaggle Datasets")


In [ ]:
OUTPUT_DIR = os.path.join(BASE_DIR, "processed")
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Save .npy files
np.save(os.path.join(OUTPUT_DIR, "X_train.npy"), X_train)
np.save(os.path.join(OUTPUT_DIR, "y_train.npy"), y_train)
np.save(os.path.join(OUTPUT_DIR, "X_test.npy"),  X_test)
np.save(os.path.join(OUTPUT_DIR, "y_test.npy"),  y_test)

# Save normalisation stats (critical for inference on new EEG)
np.save(os.path.join(OUTPUT_DIR, "train_mean.npy"), train_mean)
np.save(os.path.join(OUTPUT_DIR, "train_std.npy"),  train_std)
np.save(os.path.join(OUTPUT_DIR, "channel_names.npy"), np.array(ch_names))

# Save metadata
with open(os.path.join(OUTPUT_DIR, "info.txt"), "w") as f:
    f.write(f"Train patients ({len(TRAIN_PATIENTS)}): {TRAIN_PATIENTS}
")
    f.write(f"Test patients  ({len(TEST_PATIENTS)}): {TEST_PATIENTS}
")
    f.write(f"Window: {WINDOW_SIZE_SEC}s, Overlap: {OVERLAP}
")
    f.write(f"Non-seizure samples: {NON_SEIZURE_SAMPLES}
")
    f.write(f"Random seed: {RANDOM_SEED}
")
    f.write(f"Max recordings per patient: {MAX_RECORDINGS_PER_PATIENT or 'ALL'}
")
    f.write(f"Train shape: {X_train.shape}
")
    f.write(f"Test shape:  {X_test.shape}
")
    f.write(f"Normalization: per-channel z-score using train_mean.npy / train_std.npy\
")
    f.write(f"Channels ({len(ch_names)}): {ch_names}\
")

# Print file sizes
for fn in sorted(os.listdir(OUTPUT_DIR)):
    sz = os.path.getsize(os.path.join(OUTPUT_DIR, fn))
    print(f"  {fn}: {sz/1e6:.1f} MB")

total = sum(os.path.getsize(os.path.join(OUTPUT_DIR, f))
            for f in os.listdir(OUTPUT_DIR))
print(f"
Total: {total/1e6:.1f} MB saved to:")
print(f"  {OUTPUT_DIR}/")
print(f"
NEXT: Download from Drive -> Upload to Kaggle Datasets")

## Cell 13: DVC -- Track & Push Dataset (Versioning)

Runs once after saving .npy files. Commits this dataset version to Git + pushes data to Google Drive remote. You can roll back to any version later with `dvc checkout`.

In [6]:
!pip install -q dvc[gdrive]
!git config --global user.email "f24ce245@ms.pict.edu"
!git config --global user.name "Aswathi Pillai"

%cd /content/drive/MyDrive/EEG_PROJECT

import os
if not os.path.exists(".dvc"):
    !dvc init
    !git init --initial-branch=main
    !git add .dvc .gitignore
    !git commit -m "init dvc"
    print("DVC initialized.")
else:
    print("DVC already initialized.")

# ALWAYS set/overwrite remote (runs whether init was fresh or existed before)
!dvc remote remove myremote 2>/dev/null
!dvc remote add -d myremote /content/drive/MyDrive/EEG_PROJECT/.dvc/cache

!dvc add processed/X_train.npy processed/y_train.npy processed/X_test.npy processed/y_test.npy processed/train_mean.npy processed/train_std.npy processed/channel_names.npy processed/info.txt
!git add processed/.gitignore processed/*.dvc
!git commit -m "dataset: preprocessed EEG windows"
!dvc push

print("
Done.")
!dvc status
!dvc remote list

/content/drive/MyDrive/EEG_PROJECT
DVC already initialized.
Setting 'myremote' as a default remote.
⠋ Checking graph
Adding...:   0% 0/5 [00:00<?, ?file/s{'info': ' processed/X_train.npy |'}]
!
          |0.00 [00:00,     ?file/s]
                                    
!
  0% |          |0/? [00:00<?,    ?files/s]
                                           
  0% 0/1 [00:00<?, ?files/s]
  0% 0/1 [00:00<?, ?files/s{'info': ''}]
100% 1/1 [00:00<00:00,  1.65files/s{'info': ''}]
 20% 1/5 [00:01<00:04,  1.20s/file{'info': ' processed/y_train.npy |'}]
!
          |0.00 [00:00,     ?file/s]
                                    
!
  0% |          |0/? [00:00<?,    ?files/s]
                                           
  0% 0/1 [00:00<?, ?files/s]
  0% 0/1 [00:00<?, ?files/s{'info': ''}]
 40% 2/5 [00:01<00:01,  1.71file/s{'info': ' processed/X_test.npy |'}] 
!
          |0.00 [00:00,     ?file/s]
                                    
!
  0% |          |0/? [00:00<?,    ?files/s]
                     

In [5]:
!dvc remote list

myremote        gdrive://MyDrive/EEG_PROJECT/.dvc/cache (default)


In [7]:
!dvc status          # should say "up to date"
!dvc remote list     # should show "myremote"

Data and pipelines are up to date.
myremote        /content/drive/MyDrive/EEG_PROJECT/.dvc/cache   (default)
